# Validacao ROSE no Google Colab

**ROSE:** Robust Online Self-Adjusting Ensemble

**Paper:** Cano & Krawczyk (2022) - Machine Learning, Vol. 111(7), pp. 2561-2599

**GitHub:** https://github.com/canoalberto/ROSE

---

Este notebook valida a execucao do ROSE usando o JAR original via MOA.

## 1. Setup Ambiente

In [ ]:
# 1.1 Instalar Java (necessario para MOA)
!apt-get update -qq
!apt-get install -y default-jdk -qq

# Verificar instalacao
!java -version

In [ ]:
# 1.2 Baixar JARs do ROSE do GitHub
import urllib.request
from pathlib import Path

rose_dir = Path('rose_jars')
rose_dir.mkdir(exist_ok=True)

base_url = "https://github.com/canoalberto/ROSE/raw/master"
jars = [
    "ROSE-1.0.jar",
    "MOA-dependencies.jar",
    "sizeofag-1.0.4.jar"
]

for jar in jars:
    jar_path = rose_dir / jar
    if not jar_path.exists():
        url = f"{base_url}/{jar}"
        print(f"Baixando {jar}...")
        urllib.request.urlretrieve(url, jar_path)
        size_mb = jar_path.stat().st_size / (1024 * 1024)
        print(f"  Concluido: {size_mb:.2f} MB")
    else:
        print(f"JAR ja existe: {jar}")

print("\nJARs disponiveis:")
!ls -la rose_jars/

In [ ]:
# 1.3 Imports
import numpy as np
import pandas as pd
import subprocess
import time
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, f1_score

print("Imports concluidos")

## 2. Funcoes Auxiliares

In [ ]:
def create_arff_file(X, y, output_path, relation_name="stream"):
    """
    Cria arquivo ARFF a partir de dados numpy.

    Args:
        X: Features (n_samples, n_features)
        y: Labels (n_samples,)
        output_path: Caminho do arquivo ARFF
        relation_name: Nome da relacao
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    n_samples, n_features = X.shape
    classes = sorted(np.unique(y))

    with open(output_path, 'w') as f:
        # Header
        f.write(f"@relation {relation_name}\n\n")

        # Atributos
        for i in range(n_features):
            f.write(f"@attribute attr{i} numeric\n")

        # Classe
        class_str = ",".join([str(c) for c in classes])
        f.write(f"@attribute class {{{class_str}}}\n\n")

        # Dados
        f.write("@data\n")
        for i in range(n_samples):
            row = ",".join([str(x) for x in X[i]]) + f",{y[i]}\n"
            f.write(row)

    return output_path


def run_rose(arff_file, output_dir, max_instances=None, chunk_size=500):
    """
    Executa ROSE via MOA.

    Args:
        arff_file: Caminho do arquivo ARFF
        output_dir: Diretorio de saida
        max_instances: Maximo de instancias
        chunk_size: Frequencia de avaliacao

    Returns:
        (sucesso, resultados)
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / "rose_results.csv"
    log_file = output_dir / "rose_log.txt"

    # Classpath (Colab usa :)
    classpath = "rose_jars/ROSE-1.0.jar:rose_jars/MOA-dependencies.jar"

    # Comando
    cmd = [
        "java",
        "-Xmx4g",
        f"-javaagent:rose_jars/sizeofag-1.0.4.jar",
        "-cp", classpath,
        "moa.DoTask"
    ]

    # Task string
    task_parts = [
        "EvaluateInterleavedTestThenTrain",
        "-e", "(WindowAUCImbalancedPerformanceEvaluator)",
        "-s", f"(ArffFileStream -f {arff_file})",
        "-l", "(moa.classifiers.meta.imbalanced.ROSE)",
        "-f", str(chunk_size),
        "-d", str(output_file)
    ]

    if max_instances:
        task_parts.extend(["-i", str(max_instances)])

    task_string = " ".join(task_parts)
    cmd.append(task_string)

    print(f"Executando ROSE...")
    print(f"Comando: {' '.join(cmd[:6])} [task_string]")

    start_time = time.time()

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=600
        )

        duration = time.time() - start_time

        # Salvar logs
        with open(log_file, 'w') as f:
            f.write("=== STDOUT ===\n")
            f.write(result.stdout)
            f.write("\n\n=== STDERR ===\n")
            f.write(result.stderr)

        if result.returncode != 0:
            print(f"ROSE falhou (returncode={result.returncode})")
            print(f"Stderr: {result.stderr[:500]}")
            return False, {}

        print(f"ROSE executado com sucesso em {duration:.1f}s")

        # Parsear resultados
        results = parse_rose_results(output_file)
        results['execution_time'] = duration

        return True, results

    except subprocess.TimeoutExpired:
        print(f"ROSE timeout")
        return False, {}

    except Exception as e:
        print(f"Erro ao executar ROSE: {e}")
        import traceback
        traceback.print_exc()
        return False, {}


def parse_rose_results(results_file):
    """
    Parseia resultados do ROSE.

    CORRIGIDO: 
    - Filtro para "Learner" (nao "learning")
    - Valores ja estao em decimal (0-1), nao dividir por 100
    - Nomes corretos das colunas: Accuracy, G-Mean, pAUC, Kappa

    Args:
        results_file: Arquivo CSV de resultados

    Returns:
        Dicionario com metricas
    """
    results = {}
    results_file = Path(results_file)

    if not results_file.exists():
        print(f"Arquivo de resultados nao encontrado: {results_file}")
        return results

    try:
        with open(results_file, 'r') as f:
            lines = f.readlines()

        # Filtrar headers duplicados (comecam com "Learner" ou "learning")
        data_lines = [line for line in lines
                      if not line.startswith('Learner')
                      and not line.startswith('learning')
                      and line.strip()]

        if len(data_lines) > 0:
            header_line = lines[0]
            last_data_line = data_lines[-1]

            headers = header_line.strip().split(',')
            values = last_data_line.strip().split(',')

            data_dict = dict(zip(headers, values))

            # Debug: mostrar colunas disponiveis
            print(f"Colunas disponiveis: {list(data_dict.keys())[:10]}...")

            # Extrair metricas (valores ja estao em decimal 0-1)
            # G-mean
            if 'G-Mean' in data_dict:
                try:
                    results['gmean'] = float(data_dict['G-Mean'])
                except:
                    pass

            # pAUC (AUC periodico)
            if 'pAUC' in data_dict:
                try:
                    results['auc'] = float(data_dict['pAUC'])
                except:
                    pass

            # Accuracy (ja em decimal)
            if 'Accuracy' in data_dict:
                try:
                    results['accuracy'] = float(data_dict['Accuracy'])
                except:
                    pass

            # Kappa (ja em decimal)
            if 'Kappa' in data_dict:
                try:
                    results['kappa'] = float(data_dict['Kappa'])
                except:
                    pass

            # Recall
            if 'Recall' in data_dict:
                try:
                    results['recall'] = float(data_dict['Recall'])
                except:
                    pass

            print(f"Metricas extraidas: {results}")

    except Exception as e:
        print(f"Erro ao ler resultados: {e}")

    # Defaults
    if 'accuracy' not in results:
        results['accuracy'] = 0.0
    if 'gmean' not in results:
        results['gmean'] = results.get('accuracy', 0.0)

    return results


print("Funcoes auxiliares definidas")

## 3. Teste com Dados Sinteticos

In [ ]:
# 3.1 Gerar dados sinteticos com desbalanceamento
print("Gerando dados sinteticos...")

# Parametros
n_samples = 5000
n_features = 10
n_classes = 2
imbalance_ratio = 0.1  # 10% classe minoritaria

X, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=5,
    n_redundant=2,
    n_classes=n_classes,
    weights=[1 - imbalance_ratio, imbalance_ratio],
    random_state=42,
    flip_y=0.1
)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Distribuicao de classes: {np.bincount(y)}")
print(f"Proporcao classe 1: {np.mean(y):.2%}")

In [ ]:
# 3.2 Criar arquivo ARFF
arff_path = Path("test_data/synthetic_imbalanced.arff")
create_arff_file(X, y, arff_path, relation_name="synthetic_imbalanced")

print(f"Arquivo ARFF criado: {arff_path}")
print(f"Tamanho: {arff_path.stat().st_size / 1024:.1f} KB")

# Mostrar primeiras linhas
print("\nPrimeiras 15 linhas:")
!head -15 test_data/synthetic_imbalanced.arff

In [ ]:
# 3.3 Executar ROSE
output_dir = Path("rose_output")

success, results = run_rose(
    arff_file=arff_path,
    output_dir=output_dir,
    max_instances=n_samples,
    chunk_size=500
)

print(f"\n=== RESULTADOS ===")
print(f"Sucesso: {success}")
if success:
    print(f"Accuracy: {results.get('accuracy', 0):.4f}")
    print(f"G-mean: {results.get('gmean', 0):.4f}")
    print(f"AUC: {results.get('auc', 0):.4f}")
    print(f"Kappa: {results.get('kappa', 0):.4f}")
    print(f"Tempo: {results.get('execution_time', 0):.1f}s")

In [ ]:
# 3.4 Verificar logs
print("=== LOG DE EXECUCAO ===")
!cat rose_output/rose_log.txt | head -50

In [ ]:
# 3.5 Verificar CSV de resultados
print("=== CSV DE RESULTADOS ===")
!cat rose_output/rose_results.csv | head -20

## 4. Teste com Concept Drift

In [ ]:
# 4.1 Gerar dados com concept drift (mudanca de distribuicao)
print("Gerando dados com concept drift...")

# Conceito 1
X1, y1 = make_classification(
    n_samples=2500,
    n_features=10,
    n_informative=5,
    n_classes=2,
    weights=[0.9, 0.1],
    random_state=42,
    flip_y=0.05
)

# Conceito 2 (drift)
X2, y2 = make_classification(
    n_samples=2500,
    n_features=10,
    n_informative=5,
    n_classes=2,
    weights=[0.7, 0.3],  # Mudanca no balanceamento
    random_state=123,
    flip_y=0.05
)

# Concatenar (stream com drift no meio)
X_drift = np.vstack([X1, X2])
y_drift = np.concatenate([y1, y2])

print(f"X shape: {X_drift.shape}")
print(f"Conceito 1 - Proporcao classe 1: {np.mean(y1):.2%}")
print(f"Conceito 2 - Proporcao classe 1: {np.mean(y2):.2%}")
print(f"Total - Proporcao classe 1: {np.mean(y_drift):.2%}")

In [ ]:
# 4.2 Criar ARFF e executar ROSE
arff_drift_path = Path("test_data/synthetic_drift.arff")
create_arff_file(X_drift, y_drift, arff_drift_path, relation_name="synthetic_drift")

print(f"Arquivo ARFF criado: {arff_drift_path}")

# Executar ROSE
output_dir_drift = Path("rose_output_drift")

success_drift, results_drift = run_rose(
    arff_file=arff_drift_path,
    output_dir=output_dir_drift,
    max_instances=5000,
    chunk_size=500
)

print(f"\n=== RESULTADOS COM DRIFT ===")
print(f"Sucesso: {success_drift}")
if success_drift:
    print(f"Accuracy: {results_drift.get('accuracy', 0):.4f}")
    print(f"G-mean: {results_drift.get('gmean', 0):.4f}")
    print(f"AUC: {results_drift.get('auc', 0):.4f}")
    print(f"Tempo: {results_drift.get('execution_time', 0):.1f}s")

## 5. Comparacao com Baseline

In [ ]:
# 5.1 Executar baseline simples (Hoeffding Tree via MOA)
print("Executando baseline (Hoeffding Tree)...")

def run_baseline(arff_file, output_dir, max_instances=None, chunk_size=500):
    """Executa Hoeffding Tree como baseline."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / "baseline_results.csv"

    classpath = "rose_jars/MOA-dependencies.jar"

    cmd = [
        "java",
        "-Xmx4g",
        "-cp", classpath,
        "moa.DoTask"
    ]

    task_parts = [
        "EvaluateInterleavedTestThenTrain",
        "-e", "(WindowAUCImbalancedPerformanceEvaluator)",
        "-s", f"(ArffFileStream -f {arff_file})",
        "-l", "(trees.HoeffdingTree)",
        "-f", str(chunk_size),
        "-d", str(output_file)
    ]

    if max_instances:
        task_parts.extend(["-i", str(max_instances)])

    task_string = " ".join(task_parts)
    cmd.append(task_string)

    start_time = time.time()

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=300
        )

        duration = time.time() - start_time

        if result.returncode != 0:
            print(f"Baseline falhou")
            return False, {}

        results = parse_rose_results(output_file)
        results['execution_time'] = duration

        return True, results

    except Exception as e:
        print(f"Erro: {e}")
        return False, {}

# Executar baseline no dataset com drift
success_base, results_base = run_baseline(
    arff_file=arff_drift_path,
    output_dir="baseline_output_drift",
    max_instances=5000,
    chunk_size=500
)

print(f"\n=== BASELINE (HoeffdingTree) ===")
print(f"Sucesso: {success_base}")
if success_base:
    print(f"Accuracy: {results_base.get('accuracy', 0):.4f}")
    print(f"G-mean: {results_base.get('gmean', 0):.4f}")

In [ ]:
# 5.2 Tabela comparativa
print("\n" + "="*60)
print("COMPARACAO: ROSE vs Baseline (Dataset com Drift)")
print("="*60)

comparison = pd.DataFrame({
    'Modelo': ['ROSE', 'Hoeffding Tree'],
    'Accuracy': [
        results_drift.get('accuracy', 0) if success_drift else 0,
        results_base.get('accuracy', 0) if success_base else 0
    ],
    'G-mean': [
        results_drift.get('gmean', 0) if success_drift else 0,
        results_base.get('gmean', 0) if success_base else 0
    ],
    'Tempo (s)': [
        results_drift.get('execution_time', 0) if success_drift else 0,
        results_base.get('execution_time', 0) if success_base else 0
    ]
})

print(comparison.to_string(index=False))

if success_drift and success_base:
    gmean_diff = results_drift.get('gmean', 0) - results_base.get('gmean', 0)
    print(f"\nDiferenca G-mean (ROSE - Baseline): {gmean_diff:+.4f}")

## 6. Sumario

In [ ]:
# Sumario final
print("="*60)
print("SUMARIO DA VALIDACAO ROSE")
print("="*60)

print("\n1. Setup:")
print("   - Java instalado via apt-get")
print("   - JARs baixados do GitHub")

print("\n2. Teste Dados Sinteticos (sem drift):")
if success:
    print(f"   - Status: OK")
    print(f"   - Accuracy: {results.get('accuracy', 0):.4f}")
    print(f"   - G-mean: {results.get('gmean', 0):.4f}")
else:
    print(f"   - Status: FALHOU")

print("\n3. Teste com Concept Drift:")
if success_drift:
    print(f"   - Status: OK")
    print(f"   - Accuracy: {results_drift.get('accuracy', 0):.4f}")
    print(f"   - G-mean: {results_drift.get('gmean', 0):.4f}")
else:
    print(f"   - Status: FALHOU")

print("\n4. Comparacao com Baseline:")
if success_drift and success_base:
    print(f"   - ROSE G-mean: {results_drift.get('gmean', 0):.4f}")
    print(f"   - Baseline G-mean: {results_base.get('gmean', 0):.4f}")
    gmean_diff = results_drift.get('gmean', 0) - results_base.get('gmean', 0)
    print(f"   - Diferenca: {gmean_diff:+.4f}")

print("\n" + "="*60)
validacao_ok = success and success_drift
print(f"VALIDACAO: {'OK - ROSE pronto para integracao' if validacao_ok else 'FALHOU'}")
print("="*60)